# DIY Copilot
## Lesson: Shaping Model Output with Generation Parameters

Generative language models don't produce a single "correct" answer – at each step they sample from a probability distribution over possible next tokens. Generation parameters let you shape that sampling process, giving you control over how focused, creative, diverse, or repetitive the output is.

For a coding assistant this matters a lot: generating code needs **focused, reproducible** output, while brainstorming names or test ideas benefits from **variety**.

In this lesson, you will learn:
* how core sampling parameters (`temperature`, `top_p`, `top_k`, `max_output_tokens`) affect the generated text
* how the `presence_penalty` and `frequency_penalty` parameters can be used to control repetition
* that not every provider/model supports every parameter – unsupported ones are dropped with a warning
* why answers get cut off and how to detect it
* how streaming creates the "typewriter effect" of coding assistants

### Set up the environment
Before we can start, we have to set up the environment.

In [ ]:
# >>> SETUP <<<
# In Google Colab, this cell
#   1. clones the workshop repository,
#   2. installs its dependencies into the Python environment of this notebook (with uv),
#   3. makes the shared code (lib) importable.
# Locally (after `uv sync` in the repository), only step 3 is needed.
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/openknowledge/diy-copilot-bootcamp"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    REPO_DIR = Path("/content/diy-copilot")
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
    subprocess.run([sys.executable, str(REPO_DIR / "setup" / "install_dependencies.py")], check=True)
else:
    # The repository root is the first parent directory containing the shared code
    REPO_DIR = next(p for p in Path.cwd().resolve().parents if (p / "lib").is_dir() and (p / "pyproject.toml").exists())

# The repository is not an installed package, so we add it to the module search path
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print(f"Workshop repository: {REPO_DIR}")

In [2]:
from litellm import completion

from lib.completion import generate_completion
from lib.environment import load_environment
from lib.retry import call_with_retry

# Load the API key and the default models. If you want, you can change the model.
default_models = load_environment()
GENERATION_MODEL = default_models["generation_model"]
print(f"Model to be used: {GENERATION_MODEL}")

Model to be used: gemini/gemini-3.1-flash-lite


### Exercise 01: Models and parameters

Generation parameters control *how* a model samples its next token and thus shape the style, diversity, and length of the output. The most common ones are:

* **`temperature`** (range: `0.0` – `2.0`, typical: `0.0` – `1.0`): scales the probability distribution. Low values (→ 0) make the output focused and deterministic, high values (→ 2) make it more diverse and creative.
* **`top_p`** / nucleus sampling (range: `0.0` – `1.0`): restricts sampling to the smallest set of tokens whose cumulative probability exceeds `p`. Lower values = more focused, higher = more varied. `1.0` disables the filter.
* **`top_k`** (range: positive integer, typical: `1` – `100`): restricts sampling to the `k` most likely tokens at each step. `k = 1` is greedy decoding.
* **`max_output_tokens`** (range: positive integer, upper bound is model-dependent): caps the length of the generated response.

We ask the model for name suggestions – a typical "creative" task in a coding assistant. Run the cell several times with different parameters and observe how the suggestions change.

>**Note:** Not every model supports every parameter. Gemini models, for example, do not accept `top_k` via the OpenAI-compatible interface – our helper `generate_completion` drops such parameters and prints a warning.

In [3]:
# TODO (Optional): You can change the prompt if you like
user_prompt = (
    "Suggest 5 names for a Python function that checks whether a meeting room "
    "is free in a given time range. Answer with the names only, one per line."
)

In [4]:
# TODO: Use different generation parameters to get various answers
temperature = 0.9
top_p = 0.1
top_k = 10
max_output_tokens = 600

In [5]:
# TODO: Change the parameters and run the cell several times to see the effects
generated_text = generate_completion(
    user_prompt=user_prompt,
    model_name=GENERATION_MODEL,
    temperature=temperature,
    top_p=top_p,
    top_k=top_k,
    max_output_tokens=max_output_tokens,
)
print(f"Answer:\n{generated_text}")

Answer:
is_room_available
check_room_availability
is_meeting_room_free
is_time_slot_open
is_room_vacant


### Exercise 02: Reproducible code generation
When a coding assistant generates code, we want the same request to produce (nearly) the same result – otherwise every retry gives you a different implementation to review. Generate the same function three times, once with a **high** and once with a **low** temperature, and compare the results.

>**Note:** The effect depends on the task. For very simple tasks (e.g. "is this datetime on a quarter hour?") there is basically only one good answer, so even a high temperature produces the same code. Open tasks with many possible solutions show the effect best.

>**Note:** Even with `temperature = 0.0` the output is not guaranteed to be 100% identical – the model servers are not fully deterministic, and providers may change models behind the same name.

In [14]:
# An open task with many possible solutions
code_prompt = (
    "Write a Python function that suggests alternative free time slots when a meeting room "
    "is already booked. Include a docstring. Answer with code only."
)

In [17]:
# TODO: Generate the function three times with a high and three times with a low temperature
for temperature in [1.5, 0.0]:
    generated_codes = []
    for i in range(3):
        generated_code = generate_completion(
            user_prompt=code_prompt,
            model_name=GENERATION_MODEL,
            temperature=temperature,
            max_output_tokens=1500,
        )
        generated_codes.append(generated_code)
    print(f"==================== temperature = {temperature} ====================")
    print(f"Out of 3 results, {len(set(generated_codes))} different responses were generated. Lengths of responses: {[len(c) for c in generated_codes]}\n")
    print(f"--- Run 1 ---\n{generated_codes[0]}\n")

==================== temperature = 1.5 ====================
Out of 3 results, 3 different responses were generated. Lengths of responses: [1374, 1516, 1676]

--- Run 1 ---
```python
from datetime import datetime, timedelta

def suggest_free_slots(booked_slots, work_start="09:00", work_end="17:00", duration_minutes=60):
    """
    Suggests available time slots of a specific duration given a list of busy intervals.
    
    :param booked_slots: List of tuples/lists, each with (start_time, end_time) as strings (HH:MM).
    :param work_start: Start of the workday (HH:MM).
    :param work_end: End of the workday (HH:MM).
    :param duration_minutes: Minimum required duration for the suggested slot.
    :return: List of tuples representing available (start, end) time slots.
    """
    def to_dt(time_str):
        return datetime.strptime(time_str, "%H:%M")

    # Convert strings to datetime objects for calculation
    start_dt = to_dt(work_start)
    end_dt = to_dt(work_end)
    busy_inter

### Exercise 03: Presence and frequency penalty

The `presence_penalty` and `frequency_penalty` parameters both discourage repetition, but in different ways. Both range from `-2.0` to `2.0`:

* **`presence_penalty`**: applies a **flat** penalty to any token that has already appeared – regardless of how often. It pushes the model to introduce *new* words.
* **`frequency_penalty`**: scales the penalty with the **number of occurrences** of a token so far. It pushes the model to *stop reusing* words it has already used a lot.
* **Positive values** discourage repetition, **negative values** encourage it, **0.0** means no penalty.

The prompt below naturally invites repetition of a few key words (e.g. *room*, *booking*). Try different combinations of `presence_penalty` and `frequency_penalty` and compare how the vocabulary changes.

>**Hint:** Vary the two penalties *one at a time* (keep the other at `0.0`). Since they target different signals – presence broadens vocabulary, frequency dampens overused words – changing both at once makes it hard to tell which parameter caused the observed effect.

>**Note:** Not every model supports these parameters. If you see a warning that a parameter was dropped, the model simply ignores it. This is a good reminder that LiteLLM unifies the *interface*, but not the *capabilities* of the models.

In [8]:
# Predefined prompt: the topic naturally leads to repetition of the words "room" and "booking".
user_prompt = (
    "Write a short paragraph of about 5 sentences for a README that explains what a "
    "meeting room booking service does. Use simple, everyday language."
)

In [9]:
# TODO: Try different values between -2.0 and 2.0 and compare the outputs.
#       Suggested values to try: 0.0, 1.0, 2.0, -1.0
presence_penalty = 0.0
frequency_penalty = 0.0

In [10]:
generated_text = generate_completion(
    user_prompt=user_prompt,
    model_name=GENERATION_MODEL,
    temperature=0.7,
    presence_penalty=presence_penalty,
    frequency_penalty=frequency_penalty,
    max_output_tokens=300,
)
print(f"presence_penalty  = {presence_penalty}")
print(f"frequency_penalty = {frequency_penalty}\n")
print(f"Answer:\n{generated_text}")

presence_penalty  = 0.0
frequency_penalty = 0.0

Answer:
Our meeting room booking service makes it easy to find and reserve the perfect space for your team. You can quickly browse available rooms, check their amenities, and secure a time slot that works for everyone. No more double-bookings or wandering the halls looking for an empty spot to work. Whether you need a quick huddle space or a large conference room, our platform keeps everything organized in one place. It is a simple tool designed to help you spend less time planning and more time collaborating.


### Exercise 04: Truncated answers
If the answer is longer than `max_tokens`, the model simply stops – in the middle of a sentence or, even worse, in the middle of a function. A coding assistant must detect this, otherwise it would write broken code into your files.

The response tells us *why* the model stopped in the field `finish_reason`:
* `stop`: the model finished its answer
* `length`: the token limit was reached

Call the model with a very small token limit and check the `finish_reason`.

In [11]:
long_code_prompt = "Write a FastAPI endpoint that creates a meeting room booking, including a Pydantic model and validation."

# TODO: Call the model with max_tokens=50 and print the content and the finish_reason
# (call_with_retry waits and retries if the free tier rate limit is reached)
response = call_with_retry(
    completion,
    model=GENERATION_MODEL,
    messages=[{"role": "user", "content": long_code_prompt}],
    max_tokens=50,
)
print(response.choices[0].message.content)
print(f"\nfinish_reason: {response.choices[0].finish_reason}")

To implement a meeting room booking endpoint, we need to define a Pydantic model for request validation, a dependency for potential database interactions (omitted for brevity, but represented), and the FastAPI route handler.

### The Implementation

finish_reason: length


### Exercise 05: Streaming
Waiting several seconds for a complete answer feels slow. That's why all coding assistants **stream** the answer: with `stream=True` the model emits the answer chunk by chunk. Each chunk contains a small piece of text in `choices[0].delta.content`; concatenating them rebuilds the full answer.

Stream the answer to the prompt below and print every chunk as soon as it arrives.

>**Note:** A stream can also break off early, e.g. when the model is overloaded. Therefore we remember the `finish_reason` of the last chunk – a coding assistant should only use the answer if it is `stop`.

In [18]:
# TODO: Call the model with stream=True and print the chunks as they arrive
stream = call_with_retry(
    completion,
    model=GENERATION_MODEL,
    messages=[{"role": "user", "content": long_code_prompt}],
    max_tokens=3000,
    stream=True,
)

full_answer = ""
finish_reason = None
for chunk in stream:
    piece = chunk.choices[0].delta.content or ""
    full_answer += piece
    finish_reason = chunk.choices[0].finish_reason or finish_reason
    print(piece, end="", flush=True)

To create a FastAPI endpoint for meeting room bookings, we need to define a **Pydantic model** for the request body, use **`Field`** for validation (to ensure dates are valid and the room is specified), and handle potential logic errors.

### The Implementation

```python
from fastapi import FastAPI, HTTPException, status
from pydantic import BaseModel, Field, validator
from datetime import datetime
from typing import Optional

app = FastAPI()

# 1. Pydantic Model with Validation
class BookingRequest(BaseModel):
    room_id: int = Field(..., gt=0, description="The ID of the room must be positive")
    user_email: str = Field(..., example="user@example.com")
    start_time: datetime
    end_time: datetime

    # Custom validator to ensure end_time is after start_time
    @validator('end_time')
    def end_must_be_after_start(cls, v, values):
        if 'start_time' in values and v <= values['start_time']:
            raise ValueError('end_time must be after start_time')
        return v

In [13]:
# The concatenated chunks form the complete answer
print(f"\n\nThe answer has {len(full_answer)} characters, finish_reason: {finish_reason}")



The answer has 2828 characters, finish_reason: stop
